In [1]:
import sys

model_path = '/Users/rdobrien/skysurf/Zodiacal_Light_Model_Python'
if model_path not in sys.path:
    sys.path.append(model_path)

import time
import healpy as hp
import numpy as np
import zodisurf

In [2]:
lats = np.arange(-90, 90, 10) # Plot a range of latitudes
wave = 2.2
day_main = 150.0

days = [day_main]*len(lats) # All input arrays (except wavelength) must be the same length
lons = [0.0]*len(lats)

# Calculate intensities for all latitudes at once
intensities = zodisurf.get_zmod(wave, phase_type = 'kelsall', day = days, lon = lons, lat = lats)

In [3]:
# from zodisurf.zkernel_numba import warmup
# warmup('skysurf')           # compile now

In [4]:
# Set map resolution
nside = 128
npix = hp.nside2npix(nside)

# Retrieve pixel coordinates in ecliptic longitude and latitude (degrees)
pixels = np.arange(npix)
lon, lat = hp.pix2ang(nside, pixels, lonlat=True)

# Start the timer
start_time = time.time()

# Calculate zodiacal light values across all pixels
# (Note: If get_zmod is not natively vectorized, wrap with np.vectorize or a loop)
zodi_map = zodisurf.get_zmod(
    lambda_=1.0, phase_type='skysurf', day=[200]*len(pixels), lon=lon, lat=lat, L2=True, engine = 'numba'
)

# Stop the timer
end_time = time.time()
elapsed_time = end_time - start_time

print(
    f'Successfully populated nside={nside} map ({npix} pixels) in {elapsed_time:.4f} seconds.'
)

Successfully populated nside=128 map (196608 pixels) in 7.4186 seconds.


In [5]:
rng = np.random.default_rng(7)
n = 3000
lon = rng.uniform(0, 360, n)
lat = np.degrees(np.arcsin(rng.uniform(-1, 1, n)))
day = rng.uniform(1, 12000, n)

for phase, lam in (('skysurf', 1.0), ('kelsall', 2.2)):
    for L2 in (True, False):
        a = zodisurf.get_zmod(lam, phase, day, lon, lat, L2=L2, engine='numpy')
        b = zodisurf.get_zmod(lam, phase, day, lon, lat, L2=L2, engine='numba')
        d = np.abs(a - b)
        rel = np.max(d / np.maximum(np.abs(a), 1e-30))
        print(f"{phase:8s} L2={L2!s:5s}  max|diff|={d.max():.3g}  "
              f"max rel={rel:.3g}  exact={int((a == b).sum())}/{n}")

skysurf  L2=True   max|diff|=0  max rel=0  exact=3000/3000
skysurf  L2=False  max|diff|=0  max rel=0  exact=3000/3000
kelsall  L2=True   max|diff|=0  max rel=0  exact=3000/3000
kelsall  L2=False  max|diff|=0  max rel=0  exact=3000/3000
